# 03_baseline_roster_optimizer.ipynb

Let's create a baseline roster optimizer using linear programming. 

Start with just using data for a single season.

# Expected value of a player
Per the rules of the fantasy hockey league, skaters and goalies have different point systems.

## Skaters

The scoring for skaters (forwards and defencemen) is defined as:
- Goal: 2 Points
- Assist: 1 Point.


The expected value $v$ of a skater $i$ will be defined as:
$$
\mathbb{E}[v_i] = 2 \times \mathbb{E}[G_i] + 1 \times \mathbb{E}[A_i], 
$$

with

$$
\mathbb{E}[G_i] = \text{GPG}_i \times \mathbb{E}[\text{Games Played}],
$$

and 

$$
\mathbb{E}[A_i] = \text{APG}_i \times \mathbb{E}[\text{Games Played}],
$$

where $\mathbb{E}[G]$ is the expected goals, $\mathbb{E}[A]$ is the expected assists,  $\text{GPG}$ is the goals per game in the regular season, $\text{APG}$ is the assists per game in the regular season, and $\mathbb{E}[\text{Games Played}]$ is the expected number of games player $i$'s team will play in the playoffs.

## Goalies
The scoring for goalies is defined as:
- Win: 1 Point
- Assist: 1 Point
- Shutout: 2 Points

The dataset does not include stats on goalies assists and will therefore not be included, but since goalie assists are a rare occurence this should not be a big deal.

The expected value $v$ of a goalie $i$ will be defined as:

$$
\mathbb{E}[v_i] = 2 \times \mathbb{E}[\text{SO}_i] + 1 \times \mathbb{E}[W_i], 
$$

with

$$
\mathbb{E}[\text{SO}_i] = \text{SOPG}_i \times \mathbb{E}[\text{Games Played}],
$$

and 

$$
\mathbb{E}[W_i] = \text{WPG}_i \times \mathbb{E}[\text{Games Played}],
$$

where $\mathbb{E}[\text{SO}]$ is the expected shutouts, $\mathbb{E}[W]$ is the expected wins,  $\text{SOPG}$ is the shutouts per game in the regular season, and $\text{WPG}$ is the wins per game in the regular season.

# The LP set-up
The roster is selected once at the beginning of the playoffs.

- 15 skaters total
  - 9 forwards
  - 6 defence
- 2 goalies

This presents a straightforward linear programming problem.

## Objective 
We will simplify the above notation and just call $v_i$ the expected value of player $i$.

Let $x_i=1$ if you pick player $i$, otherwise $x_i=0$. 

The objective function is then simply

$$
\text{max}\sum_i v_ix_i.
$$

## Constraints
The number of skaters in each position are our constraints, which can be written out as follows:

$$
\text{Total number of players}: \sum_i x_i = 17,
$$

$$
\text{Total number of forwards}: \sum_{i \in \text{forwards}} x_i = 9,
$$

$$
\text{Total number of defence}: \sum_{i \in \text{defence}} x_i = 6,
$$

$$
\text{Total number of goalies}: \sum_{i \in \text{goalies}} x_i = 2.
$$

We can also add one strategic constraint to limit the number of players we select per team as a way to hedge our roster selections (in case of a bracket upset for example):

$$
\text{Limit per team}: \sum_{i \in \text{Team T}} x_i \leq M_{\text{T}} \quad \forall \quad \text{T},
$$

where $M_{\text{T}}$ is our defined maximum number of players for a given team, $\text{T}$.

# The Baseline Plan

- For skaters, compute the goals and assists per game in the regular season. 
- For goalies, compute the wins and shutouts per game in the regular season.

We will make the naive assumption that these rates continue in the playoffs. This assumption is what makes this a baseline model. In future versions, the expected value of a player in the playoffs will be the output of a machine learning model. 

- Multiply the rates by an estimate of how many games the given team's expected number of games played in the playoffs to get the expected value per player.

For a baseline implementation, maybe I could just use a sportsbook's predictions on playoff outcomes to estimate what round each team will make in the playoffs and then make an assumption that each series lasts say 6 games.

Or for running this LP problem on previous seasons I can just use the actual number of games each team played.

With the expected value per player calculated, we can then use scipy to solve the LP problem.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
from scipy.optimize import milp, Bounds, LinearConstraint

BASE_PATH = Path(r"..\data\natural_stat_trick\raw")

def ffn_from_params(params):
    fpath = os.path.join(BASE_PATH, params["year"], params["season"])
    ffn = os.path.join(fpath, f"{params["category"]}.csv")
    return ffn

# Load data

In [3]:
params = {
    "year": "2022",
    "season": "regular",
    "category": "skaters"
    }

ffn = ffn_from_params(params)

skaters = pd.read_csv(ffn)
print(skaters.info())
skaters.head()

<class 'pandas.DataFrame'>
RangeIndex: 951 entries, 0 to 950
Data columns (total 35 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        951 non-null    int64  
 1   Player            951 non-null    str    
 2   Team              951 non-null    str    
 3   Position          951 non-null    str    
 4   GP                951 non-null    int64  
 5   TOI               951 non-null    float64
 6   Goals             951 non-null    int64  
 7   Total Assists     951 non-null    int64  
 8   First Assists     951 non-null    int64  
 9   Second Assists    951 non-null    int64  
 10  Total Points      951 non-null    int64  
 11  IPP               951 non-null    str    
 12  Shots             951 non-null    int64  
 13  SH%               951 non-null    str    
 14  ixG               951 non-null    float64
 15  iCF               951 non-null    int64  
 16  iFF               951 non-null    int64  
 17  iSCF    

,Unnamed: 0,Player,Team,Position,GP,TOI,Goals,Total Assists,First Assists,Second Assists,...,Misconduct,Penalties Drawn,Giveaways,Takeaways,Hits,Hits Taken,Shots Blocked,Faceoffs Won,Faceoffs Lost,Faceoffs %
0,438,Connor McDavid,EDM,C,82,1835.616667,64,89,60,29,...,0,45,77,82,89,108,37,525,486,51.93
1,367,Leon Draisaitl,EDM,C,80,1738.716667,52,76,60,16,...,0,22,102,77,66,134,36,807,663,54.90
2,207,Nikita Kucherov,T.B,R,82,1650.850000,30,83,48,35,...,0,38,98,57,61,107,6,2,0,100.00
3,384,David Pastrnak,BOS,R,82,1604.283333,61,52,31,21,...,0,29,109,52,91,130,31,8,11,42.11
4,334,Nathan MacKinnon,COL,C,71,1584.533333,42,69,42,27,...,0,33,47,43,53,73,33,519,649,44.43


In [4]:
params = {
    "year": "2022",
    "season": "regular",
    "category": "goalies"
    }

ffn = ffn_from_params(params)

goalies = pd.read_csv(ffn)
print(goalies.info())
goalies.head()

<class 'pandas.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Unnamed: 0                107 non-null    int64  
 1   Player                    107 non-null    str    
 2   Team                      107 non-null    str    
 3   GP                        107 non-null    int64  
 4   TOI                       107 non-null    float64
 5   Shots Against             107 non-null    int64  
 6   Saves                     107 non-null    int64  
 7   Goals Against             107 non-null    int64  
 8   SV%                       107 non-null    str    
 9   GAA                       107 non-null    float64
 10  GSAA                      107 non-null    float64
 11  xG Against                107 non-null    float64
 12  HD Shots Against          107 non-null    int64  
 13  HD Saves                  107 non-null    int64  
 14  HD Goals Against     

,Unnamed: 0,Player,Team,GP,TOI,Shots Against,Saves,Goals Against,SV%,GAA,...,LD Shots Against,LD Saves,LD Goals Against,LDSV%,LDGAA,LDGSAA,Rush Attempts Against,Rebound Attempts Against,Avg. Shot Distance,Avg. Goal Distance
0,104,Matthew Berlin,EDM,1,2.433333,1,1,0,1.000,0.00,...,1,1,0,1.000,0.0,0.04,0,0,51.00,-
1,98,Dustin Wolf,CGY,1,59.900000,24,23,1,0.958,1.00,...,8,8,0,1.000,0.0,0.31,3,4,47.38,7.00
2,24,Keith Kinkaid,"BOS, COL",2,87.850000,40,38,2,0.950,1.37,...,16,16,0,1.000,0.0,0.62,1,7,36.30,10.00
3,80,Dylan Ferguson,OTT,2,118.866667,83,78,5,0.940,2.52,...,29,29,0,1.000,0.0,1.13,2,15,34.04,18.20
4,103,Jet Greaves,CBJ,1,58.950000,49,46,3,0.939,3.05,...,13,13,0,1.000,0.0,0.51,4,6,27.96,14.00


# Expected value of players


